<a href="https://colab.research.google.com/github/Rageel-28/244107020136-Machine-Learning/blob/main/JS09/JS09_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS09 - Approximate Nearest Neighbors (ANN)
## Lab 4 - ANNOY vs. FAISS vs. HNSW

Percobaan ini membandingkan tiga model pada data yang jauh lebih besar: **1.000.000 data, 5 dimensi**, mencari **10 tetangga terdekat**.

| Model | Konfigurasi |
|---|---|
| ANNOY | `euclidean`, 10 pohon |
| FAISS | `IndexFlatL2` (*exact*, tanpa aproksimasi: dipakai sebagai pembanding/*ground truth*) |
| HNSW | `space='l2'`, `M=16`, `ef_construction=200`, `ef=50` |

> **Waktu proses:** membangun indeks HNSW untuk 1 juta data dapat memakan beberapa menit (modul melaporkan ±157 detik). Jika memori/waktu terbatas, ubah `n_data` menjadi `100_000`.

In [1]:
# Instal library bila belum ada (di Colab setara dengan !pip install ...)
import importlib, subprocess, sys

for modul, paket in [('annoy', 'annoy'), ('faiss', 'faiss-cpu'), ('hnswlib', 'hnswlib')]:
    try:
        importlib.import_module(modul)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paket])

print('Library siap dipakai')

Library siap dipakai


In [2]:
import numpy as np
import time
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

## Langkah 1 - Membuat Data (1 juta baris, 5 dimensi)

In [3]:
# ===============================
# Build 1mio rows with 5D
# ===============================
np.random.seed(42)
n_data = 1_000_000   # try 100_000 if you have limited memory
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10
print('Ukuran data:', X.shape)

Ukuran data: (1000000, 5)


## Langkah 2 - ANNOY

In [4]:
# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

waktu = {'ANNOY': (build_time, query_time)}
hasil_idx = {'ANNOY': np.array(neighbors[0])}

=== Annoy ===
Build time: 25.904609441757202 detik
Query time: 0.0001876354217529297 detik
Neighbors: [901095, 495168, 561017, 163986, 185734] ...


## Langkah 3 - FAISS (Flat Index)

In [5]:
# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

waktu['FAISS Flat (exact)'] = (build_time, query_time)
hasil_idx['FAISS Flat (exact)'] = indices[0]

=== FAISS (IndexFlatL2) ===
Build time: 0.015022516250610352 detik
Query time: 0.009020328521728516 detik
Neighbors: [901095 495168 561017 163986 185734] ...


## Langkah 4 - HNSW (hnswlib)

In [6]:
# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

waktu['HNSW'] = (build_time, query_time)
hasil_idx['HNSW'] = labels[0]

=== HNSW (hnswlib) ===
Build time: 166.71422958374023 detik
Query time: 0.00021719932556152344 detik
Neighbors: [901095 495168 561017 163986 185734] ...


## Langkah 5 - Ringkasan dan Recall (1 Query)

`IndexFlatL2` bersifat *exact*, sehingga hasilnya dipakai sebagai *ground truth* untuk menghitung *recall@10* dua model aproksimasi.

In [7]:
gt = set(hasil_idx['FAISS Flat (exact)'])
baris = []
for nama, (b, q) in waktu.items():
    rec = len(set(hasil_idx[nama]) & gt) / k
    baris.append({'Model': nama, 'Build (detik)': b, 'Query (ms)': q * 1000, 'Recall@10 (1 query)': rec})
pd.DataFrame(baris).round(4)

,Model,Build (detik),Query (ms),Recall@10 (1 query)
0,ANNOY,25.9046,0.1876,1.0
1,FAISS Flat (exact),0.0150,9.0203,1.0
2,HNSW,166.7142,0.2172,1.0


## Tambahan - 100 Query Acak

Satu *query* dan satu kali pengukuran waktu tidak stabil. Berikut rata-rata dari **100 *query* acak** (waktu dalam ms per *query*), dengan *recall@10* terhadap `IndexFlatL2`.

In [8]:
rng = np.random.RandomState(1)
Q = rng.random_sample((100, dim)).astype(np.float32)

# Ground truth + waktu exact
t0 = time.perf_counter()
_, I_gt = faiss_index.search(Q, k)
t_flat = (time.perf_counter() - t0) * 1000 / len(Q)

# ANNOY (per query, dalam loop)
t0 = time.perf_counter()
I_ann = np.array([ann_index.get_nns_by_vector(q, k) for q in Q])
t_ann = (time.perf_counter() - t0) * 1000 / len(Q)

# HNSW (batch)
t0 = time.perf_counter()
I_hnsw, _ = hnsw_index.knn_query(Q, k=k)
t_hnsw = (time.perf_counter() - t0) * 1000 / len(Q)


def recall(I_pred, I_true):
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(I_pred, I_true)])


tabel100 = pd.DataFrame([
    {'Model': 'FAISS Flat (exact)', 'Query (ms/query)': t_flat, 'Recall@10': 1.0},
    {'Model': 'ANNOY (10 trees)',   'Query (ms/query)': t_ann, 'Recall@10': recall(I_ann, I_gt)},
    {'Model': 'HNSW (ef=50)',       'Query (ms/query)': t_hnsw, 'Recall@10': recall(I_hnsw, I_gt)},
])
tabel100.round(4)

,Model,Query (ms/query),Recall@10
0,FAISS Flat (exact),5.0482,1.000
1,ANNOY (10 trees),0.0781,0.942
2,HNSW (ef=50),0.0424,1.000


In [9]:
# Berapa kali lebih cepat dibanding exact?
for _, r in tabel100.iterrows():
    print(f"{r['Model']:<20} {r['Query (ms/query)']:8.4f} ms/query  | kecepatan relatif terhadap Flat: {t_flat / r['Query (ms/query)']:6.2f}x")

FAISS Flat (exact)     5.0482 ms/query  | kecepatan relatif terhadap Flat:   1.00x
ANNOY (10 trees)       0.0781 ms/query  | kecepatan relatif terhadap Flat:  64.66x
HNSW (ef=50)           0.0424 ms/query  | kecepatan relatif terhadap Flat: 119.02x


### Analisis

Angka berikut dari satu run (1.000.000 data, 5D, $k=10$); waktu bergantung mesin (modul melaporkan *build* ANNOY ±20 detik dan HNSW ±157 detik pada mesin yang berbeda).

| Model | *Build* (detik) | *Query* (ms/*query*, 100 *query*) | Recall@10 |
|---|---|---|---|
| FAISS Flat (exact) | 0.013 | 2.377 | 1.000 |
| ANNOY (10 pohon) | 8.73 | 0.109 | 0.942 |
| HNSW (`ef=50`) | 76.45 | 0.025 | 1.000 |

- **Satu *query*:** ketiganya mengembalikan tetangga teratas yang sama (recall 1.0), sehingga satu *query* saja tidak memperlihatkan perbedaan akurasi. Pada 100 *query*, ANNOY hanya menemukan ±94% tetangga sebenarnya.
- **Kecepatan *query*:** ANNOY ±22× dan HNSW ±94× lebih cepat dari pencarian *exact*.
- **Biaya *build*:** HNSW membutuhkan waktu ±9× lebih lama dari ANNOY. Biaya ini dibayar di depan, lalu dilunasi bila indeks dipakai untuk banyak *query*. Untuk data yang sering berubah atau sedikit *query*, ANNOY dapat lebih masuk akal.
- **Catatan keadilan:** FAISS dan HNSW (`knn_query` batch) memanfaatkan beberapa *thread*, sedangkan ANNOY diproses satu per satu dalam *loop* Python, sehingga sebagian selisih kecepatan berasal dari paralelisme, bukan hanya algoritma.
- **Catatan:** FAISS `IndexFlatL2` adalah pencarian *exact* (bukan ANN) dan tetap cepat berkat implementasi yang teroptimasi, tetapi biayanya naik linear terhadap jumlah data. Data 5D adalah kasus mudah bagi ANN; Lab 5 menguji kasus yang lebih sulit (128D).